
# YOLOv11n Training with Roboflow Dataset

DCC Lab - 2026

[HAM]

This notebook trains a **YOLOv11 nano** object detection model on a Roboflow dataset and exports to **ONNX** and **TensorRT-ready** formats.
Inhouse training is offered on the Roboflow platform and it is good. However, downloading weight requires account upgrade. Make it not really worth for simple task/project.

### Steps
1. Set the runtime as GPU
2. Fill the Roboflow credentials (cell 2)
3. Set the training parameters, mainly batch size and epoch (cell 2), Keep the image size as 640
4. Run all of the cells
5. Make sure the internet is not lost and regularly check the training notebook, so the training process is not interrupted.
6. When the training finished, takes the "best.pt" weight, then bring it to the Jetson Nano for ".onnx" and ".engine" conversions.

**Notes**: This notebook is also able to train another versions of YOLO, simply change the yolo versions on the cell 2 and cell 3.

In [ ]:
# Verify GPU is available
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')


Sat May 16 15:47:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install ultralytics roboflow -q

import ultralytics
ultralytics.checks()


Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.1/112.6 GB disk)


In [ ]:
# ════════════════════════════════════════════════════
# CONFIGURE YOUR ROBOFLOW DATASET AND TRAINING HERE
# ════════════════════════════════════════════════════

# get this data from your dataset/versions --> export --> download, with link options

ROBOFLOW_API_KEY   = 'API-KEY'       # <-- paste your Roboflow API key
ROBOFLOW_WORKSPACE = 'workspace-name'            # <-- e.g. 'my-team'
ROBOFLOW_PROJECT   = 'project-name'       # <-- e.g. 'smart-trash-detector'
ROBOFLOW_VERSION   = 1                            # <-- dataset version (integer)

# Training hyperparameters
EPOCHS      = 50
BATCH_SIZE  = 16     # reduce to 8 if you get OOM errors
IMAGE_SIZE  = 640    # YOLOv8 default; use 320 for faster training
MODEL       = 'yolo11n.pt'  # nano — fastest; options: yolov8s.pt, yolov8m.pt
PROJECT_DIR = 'runs/detect'
RUN_NAME    = ROBOFLOW_PROJECT

print(f'Dataset : {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT} v{ROBOFLOW_VERSION}')
print(f'Model   : {MODEL}  |  Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}  |  Imgsz: {IMAGE_SIZE}')


Dataset : ekkys-workspace/smart-trash-detector v2
Model   : yolo11n.pt  |  Epochs: 50  |  Batch: 16  |  Imgsz: 640


In [ ]:
from roboflow import Roboflow

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)

# Download in YOLOv11 format — gives us data.yaml + train/valid/test splits
dataset = version.download('yolo11', location=f'datasets/{ROBOFLOW_PROJECT}', overwrite=True)

DATA_YAML = f'{dataset.location}/data.yaml'
print(f'data.yaml: {DATA_YAML}')

# Preview the yaml
!cat {DATA_YAML}


loading Roboflow workspace...
loading Roboflow project...

Version export complete for yolo11 format



Extracting Dataset Version Zip to datasets/smart-trash-detector in yolo11:: 100%|██████████| 1643/1643 [00:00<00:00, 3320.85it/s]


data.yaml: /content/datasets/smart-trash-detector/data.yaml
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 3
names: ['Sterofoam', 'lain-lainya', 'plastikk']

roboflow:
  workspace: ekkys-workspace
  project: smart-trash-detector
  version: 2
  license: CC BY 4.0
  url: https://universe.roboflow.com/ekkys-workspace/smart-trash-detector/dataset/2

In [ ]:
import yaml
from pathlib import Path

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

base = Path(dataset.location).resolve()  # /content/datasets/smart-trash-detector

# Map Roboflow split names to actual folder names on disk
SPLIT_FOLDERS = {'train': 'train', 'val': 'valid', 'test': 'test'}

for key, folder in SPLIT_FOLDERS.items():
    resolved = base / folder / 'images'
    if resolved.exists():
        data[key] = str(resolved)
        print(f"  {key} -> {data[key]}")
    else:
        print(f"  [warn] {key} -> {resolved} NOT FOUND")

with open(DATA_YAML, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("\ndata.yaml after fix:")
!cat {DATA_YAML}

  train -> /content/datasets/smart-trash-detector/train/images
  val -> /content/datasets/smart-trash-detector/valid/images
  test -> /content/datasets/smart-trash-detector/test/images

data.yaml after fix:
names:
- Sterofoam
- lain-lainya
- plastikk
nc: 3
roboflow:
  license: CC BY 4.0
  project: smart-trash-detector
  url: https://universe.roboflow.com/ekkys-workspace/smart-trash-detector/dataset/2
  version: 2
  workspace: ekkys-workspace
test: /content/datasets/smart-trash-detector/test/images
train: /content/datasets/smart-trash-detector/train/images
val: /content/datasets/smart-trash-detector/valid/images


In [ ]:
# See what folders actually exist in the dataset
import os
for root, dirs, files in os.walk(dataset.location):
    level = root.replace(dataset.location, '').count(os.sep)
    if level < 3:
        print(' ' * level * 2 + os.path.basename(root) + '/')

smart-trash-detector/
  train/
    labels/
    images/
  valid/
    labels/
    images/
  test/
    labels/
    images/


In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)  # loads pretrained YOLOv8n weights from Ultralytics

results = model.train(
    data    = DATA_YAML,
    epochs  = EPOCHS,
    imgsz   = IMAGE_SIZE,
    batch   = BATCH_SIZE,
    project = PROJECT_DIR,
    name    = RUN_NAME,
    device  = 0,          # GPU 0; set to 'cpu' if no GPU
    plots   = True,       # saves confusion matrix, PR curve, etc.
    save    = True,
    exist_ok= True,
)

print('Training complete!')
print(f'Best weights: {PROJECT_DIR}/{RUN_NAME}/weights/best.pt')


Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/smart-trash-detector/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=smart-trash-detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ov

In [ ]:
# Load best weights and evaluate on the validation set
best_model = YOLO('/content/runs/detect/runs/detect/smart-trash-detector/weights/best.pt')

metrics = best_model.val(data=DATA_YAML, imgsz=IMAGE_SIZE, device=0)

print(f'mAP50     : {metrics.box.map50:.4f}')
print(f'mAP50-95  : {metrics.box.map:.4f}')
print(f'Precision : {metrics.box.mp:.4f}')
print(f'Recall    : {metrics.box.mr:.4f}')


Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2493.2±978.8 MB/s, size: 89.3 KB)
val: Scanning /content/datasets/smart-trash-detector/valid/labels.cache... 34 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 34/34 17.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.4it/s 2.2s
                   all         34        375      0.413      0.322      0.338      0.185
             Sterofoam         22         81      0.621      0.395       0.49      0.297
           lain-lainya         21         66      0.187      0.167      0.146     0.0903
              plastikk         33        228       0.43      0.404      0.378      0.166
Speed: 10.4ms preprocess, 8.4ms inference, 0.0ms loss, 4.8ms postprocess per image
Results saved to /content/runs/dete

In [ ]:
from IPython.display import Image, display
import glob

run_dir = f'{PROJECT_DIR}/{RUN_NAME}'

for img_path in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    full = f'{run_dir}/{img_path}'
    if glob.glob(full):
        print(f'--- {img_path} ---')
        display(Image(full))
